# 01: Original Codebase Validation

## Objective

Validate the original REPLICA implementation of unigram probability estimators.

## References

- Hopton et al. (2026)
- https://aclanthology.org/2026.emnlp-main.2399/
- https://github.com/replicaverse/replica

## Code attribution

The NMC and MMC estimators are imported directly from the REPLICA framework.

In [1]:
import sys
import torch
import transformers
import replica

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("REPLICA:", replica.__file__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/home/hydrogen/PROJECTS/replica/.venv-replica/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.12.3
Interpreter: /home/hydrogen/PROJECTS/replica/.venv-replica/bin/python
PyTorch: 2.15.0+cu132
Transformers: 4.57.6
REPLICA: /home/hydrogen/PROJECTS/replica/src/replica/__init__.py
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import math
import pandas as pd

from replica.feature.frequency.sampled import (
    mc_estimate,
    SampledUnigramSource,
)

def to_surprisal(probability):
    return -math.log2(probability) if probability > 0 else math.inf

## Naive Monte Carlo

Execution of the original NMC implementation
on controlled text samples.

This experiment verifies the behavior of the estimator
when a target word is present or absent in the samples.

In [3]:
samples = [
    "the dog ran across the field",
    "a cat sat on the mat",
    "the the the dog dog",
    "hello world this is a test",
]

words = ["the", "dog", "cat", "world", "coriander"]

nmc_results = pd.DataFrame([
    {
        "word": word,
        "probability": mc_estimate(word, samples),
    }
    for word in words
])

nmc_results["surprisal"] = (
    nmc_results["probability"].map(to_surprisal)
)

nmc_results

,word,probability,surprisal
0,the,0.193548,2.369234
1,dog,0.096774,3.369234
2,cat,0.032258,4.954196
3,world,0.032258,4.954196
4,coriander,0.000000,inf


## Marginalized Monte Carlo

Execution of the original MMC implementation using
DistilGPT2 on a CUDA-enabled device.

In [4]:
source = SampledUnigramSource(
    model_name="distilgpt2",
    n_samples=16,
    max_len=20,
    device="cuda",
    seed=42,
    batch_size=4,
    gen_batch_size=4,
)

words = [" the", " dog", " cat", " coriander"]

estimates = source.estimate_words(words)

mmc_results = pd.DataFrame([
    {
        "word": word.strip(),
        "probability": probability,
        "surprisal": to_surprisal(probability),
    }
    for word, probability in estimates.items()
])

mmc_results

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


,word,probability,surprisal
0,the,2.978624e-02,5.069210
1,dog,8.899829e-05,13.455863
2,cat,6.746232e-05,13.855558
3,coriander,1.337638e-07,22.833809
